In [ ]:
%run lib.py
%run init.ipynb

In [ ]:
import lightgbm as lgb
import xgboost as xgb

from tqdm.auto import tqdma

In [ ]:
LGB_PARAMS = {
    "objective": "regression",
    "metric": "mae",
    "verbosity": -1,
    "random_state": 42
}

XGB_PARAMS = {
    "objective": "reg:squarederror",
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.05,
    "random_state": 42
}

In [ ]:
def create_features(df):

    df = df.copy()

    df = df.sort_values(
        ["unique_id", "ds"]
    )

    for lag in [1, 2, 3, 6, 12]:

        df[f"lag_{lag}"] = (
            df.groupby("unique_id")["y"]
            .shift(lag)
        )

    df["rolling_mean_3"] = (
        df.groupby("unique_id")["y"]
        .shift(1)
        .rolling(3)
        .mean()
    )

    df["rolling_mean_6"] = (
        df.groupby("unique_id")["y"]
        .shift(1)
        .rolling(6)
        .mean()
    )

    return df

In [ ]:
def prepare_ml_data(
    group,
    horizon
):

    group = create_features(
        group
    )

    train = group.iloc[:-horizon]

    test = group.iloc[-horizon:]

    train = train.dropna()

    feature_cols = [

        c

        for c in train.columns

        if c not in [
            "unique_id",
            "ds",
            "y"
        ]
    ]

    X_train = train[
        feature_cols
    ]

    y_train = train["y"]

    return (
        X_train,
        y_train,
        test,
        feature_cols
    )

In [ ]:
def build_future_features(
    history
):

    return pd.DataFrame({

        "lag_1":
            [history[-1]],

        "lag_2":
            [history[-2]],

        "lag_3":
            [history[-3]],

        "lag_6":
            [history[-6]],

        "lag_12":
            [history[-12]],

        "rolling_mean_3":
            [np.mean(history[-3:])],

        "rolling_mean_6":
            [np.mean(history[-6:])]
    })

In [ ]:
def lightgbm_forecast(
    train_series,
    horizon
):

    group = pd.DataFrame({
        "unique_id": ["x"] * len(train_series),
        "ds": range(len(train_series)),
        "y": train_series
    })

    X_train, y_train, _, _ = (
        prepare_ml_data(
            group,
            0
        )
    )

    model = lgb.LGBMRegressor(
        **LGB_PARAMS
    )

    model.fit(
        X_train,
        y_train
    )

    history = list(
        train_series
    )

    forecasts = []

    for _ in range(horizon):

        X_future = (
            build_future_features(
                history
            )
        )

        pred = model.predict(
            X_future
        )[0]

        forecasts.append(pred)

        history.append(pred)

    return np.array(
        forecasts
    )

In [ ]:
def xgboost_forecast(
    train_series,
    horizon
):

    group = pd.DataFrame({
        "unique_id": ["x"] * len(train_series),
        "ds": range(len(train_series)),
        "y": train_series
    })

    X_train, y_train, _, _ = (
        prepare_ml_data(
            group,
            0
        )
    )

    model = xgb.XGBRegressor(
        **XGB_PARAMS
    )

    model.fit(
        X_train,
        y_train
    )

    history = list(
        train_series
    )

    forecasts = []

    for _ in range(horizon):

        X_future = (
            build_future_features(
                history
            )
        )

        pred = model.predict(
            X_future
        )[0]

        forecasts.append(pred)

        history.append(pred)

    return np.array(
        forecasts
    )


In [ ]:
results = []

MODEL_MAP = {

    "LightGBM":
        lightgbm_forecast,

    "XGBoost":
        xgboost_forecast
}

CSV_FILES = sorted(
    DATA_DIR.glob("*.csv")
)

for file in CSV_FILES:

    dataset_name = file.stem

    print()
    print("=" * 80)
    print(dataset_name)

    horizon = get_horizon(
        dataset_name
    )

    df = load_dataset(
        file
    )

    print(
        dataset_summary(df)
    )

    for model_name, model_func in MODEL_MAP.items():

        print(
            f"Running {model_name}"
        )

        start = time()

        metrics = evaluate_dataset(
            df,
            model_func,
            horizon
        )

        runtime = (
            time() - start
        )

        result = build_result_row(
            dataset_name,
            model_name,
            horizon,
            runtime,
            metrics,
            df
        )

        results.append(
            result
        )

In [ ]:
results_df = pd.DataFrame(
    results
)

results_df.head()

results_df.to_csv(
    RESULTS_DIR /
    "machine_learning_results.csv",
    index=False
)

In [ ]:
results_df.sort_values(
    ["dataset", "MAE"]
)

results_df.groupby(
    "model"
)[
    ["MAE", "RMSE", "SMAPE"]
].mean().sort_values(
    "MAE"
)